In [ ]:
#Use Llama Index to query chunked texts in AASHTO manual
import os
import re
import glob
import time
import requests
import fitz  # PyMuPDF
import gradio as gr

from typing import List, Optional

from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Settings,
)
from llama_index.core.node_parser import SentenceSplitter, SimpleNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


# -----------------------------
# 1. Global settings: embeddings
# -----------------------------

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)


# -----------------------------
# 2. Engineering-style chunking
# -----------------------------

SECTION_PATTERN = re.compile(
    r"^([A-Z]?\d+(\.\d+){0,3})\s+.+",
    re.MULTILINE,
)

def engineering_chunker(
    text: str,
    chunk_size: int = 800,
    chunk_overlap: int = 100,
) -> List[str]:
    # First split by engineering-style section headers
    sections = SECTION_PATTERN.split(text)

    blocks = []
    for i in range(1, len(sections), 3):
        header = sections[i]
        body = sections[i + 2] if i + 2 < len(sections) else ""
        blocks.append(f"{header}\n{body}")

    if not blocks:
        blocks = [text]

    splitter = SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    final_chunks = []
    for block in blocks:
        final_chunks.extend(splitter.split_text(block))

    return final_chunks


class EngineeringNodeParser(SimpleNodeParser):
    def _parse_file(self, file_path, file_text):
        chunks = engineering_chunker(file_text)
        return [self._text_to_node(chunk) for chunk in chunks]


# -----------------------------
# 3. PDF validation + retry
# -----------------------------

def is_valid_pdf(path: str) -> bool:
    try:
        with fitz.open(path) as doc:
            _ = doc.load_page(0)
        return True
    except Exception:
        return False


def download_with_retry(
    url: str,
    dest: str,
    retries: int = 3,
    timeout: int = 20,
) -> bool:
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(
                url,
                headers={"User-Agent": "Mozilla/5.0"},
                timeout=timeout,
            )
            r.raise_for_status()

            with open(dest, "wb") as f:
                f.write(r.content)

            if is_valid_pdf(dest):
                print(f"✓ Valid PDF downloaded: {dest}")
                return True
            else:
                print(f"✗ Invalid PDF detected, retrying ({attempt}/{retries})")
        except Exception as e:
            print(f"✗ Download error: {e} (attempt {attempt}/{retries})")

        time.sleep(2 ** attempt)

    print(f"✗ Failed after {retries} attempts: {url}")
    return False


# -----------------------------
# 4. Integrated parser class
# -----------------------------

class EngineeringDocumentPipeline:
    def __init__(
        self,
        root_dir: str,
        url_map: Optional[dict] = None,
    ):
        """
        root_dir: folder containing PDFs
        url_map: optional dict {filename.pdf: source_url} for auto-repair
        """
        self.root_dir = root_dir
        self.url_map = url_map or {}
        self.index: Optional[VectorStoreIndex] = None
        self.query_engine = None

    def repair_pdf_folder(self):
        pdf_paths = glob.glob(os.path.join(self.root_dir, "*.pdf"))

        for pdf in pdf_paths:
            filename = os.path.basename(pdf)

            if is_valid_pdf(pdf):
                print(f"✓ OK: {filename}")
                continue

            print(f"⚠ Corrupted PDF found: {filename}")

            url = self.url_map.get(filename)
            if not url:
                print(f"✗ No URL mapping for {filename}, skipping auto-repair.")
                continue

            success = download_with_retry(url, pdf)
            if not success:
                print(f"✗ Skipping unrecoverable file: {filename}")

    def build_index(self):
        # 1) Optionally repair PDFs
        self.repair_pdf_folder()

        # 2) Load documents
        docs = SimpleDirectoryReader(self.root_dir).load_data()

        # 3) Parse into nodes with engineering-aware chunking
        parser = EngineeringNodeParser()
        nodes = parser.get_nodes_from_documents(docs)

        # 4) Build index
        self.index = VectorStoreIndex(nodes)
        self.query_engine = self.index.as_query_engine()

    def query(self, question: str) -> str:
        if self.query_engine is None:
            raise RuntimeError("Index not built yet. Call build_index() first.")
        response = self.query_engine.query(question)
        return str(response)


# -----------------------------
# 5. Instantiate pipeline
# -----------------------------

ROOT_DIR = r"C:\Users\anaft\AI_automation_1\AASHTO_Geo_Manual"

# Optional: map filenames to their source URLs if you want auto-repair
URL_MAP = {
    # "AASHTO_Chapter1.pdf": "https://example.com/AASHTO_Chapter1.pdf",
    # "AASHTO_Chapter2.pdf": "https://example.com/AASHTO_Chapter2.pdf",
}

pipeline = EngineeringDocumentPipeline(
    root_dir=ROOT_DIR,
    url_map=URL_MAP,
)

print("Building index (this may take a while the first time)...")
pipeline.build_index()
print("Index ready.")


# -----------------------------
# 6. Gradio UI
# -----------------------------

def chat_fn(message, history):
    answer = pipeline.query(message)
    history = history + [(message, answer)]
    return "", history

with gr.Blocks() as demo:
    gr.Markdown("# Engineering Manual RAG Chat\nAsk questions about the AASHTO Geo Manual.")
    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Your question")
    clear = gr.Button("Clear")

    msg.submit(chat_fn, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: ([], ""), None, [chatbot, msg])

demo.launch()
